# nmtc-calc — NMTC Transaction Calculator
## Demo: Southside Community Health Center

This notebook models a complete New Markets Tax Credit (NMTC) leveraged
transaction for a community health center in a low-income census tract.

We walk through the full deal structure: QEI sizing, 7-year credit schedule,
investor economics, and net subsidy to the project.


In [ ]:
import sys
sys.path.insert(0, '..')

from nmtccalc import NMTCDeal, transaction, credits, investor, subsidy
import pandas as pd
pd.set_option('display.max_columns', None)


## 1. Define the Deal

In [ ]:
deal = NMTCDeal(
    project_name="Southside Community Health Center",
    total_project_cost=12_000_000,
    nmtc_allocation=10_000_000,
    credit_price=0.83,
    leverage_loan_rate=0.045,
    qlici_a_loan_rate=0.045,
    qlici_b_loan_rate=0.010,
    cde_fee_rate=0.02,
    compliance_years=7,
    discount_rate=0.08,
    investor_name="US Bancorp CDC",
    cde_name="Chicago Development Fund",
    project_location="Chicago, IL",
)

print(deal)
print(f"\nTotal NMTCs:      \${deal.total_nmtcs/1e6:.2f}MM")
print(f"Investor Equity:  \${deal.investor_equity/1e6:.2f}MM")
print(f"Leverage Loan:    \${deal.leverage_loan/1e6:.2f}MM")


## 2. Full Capital Stack Structure

Breaking down the QEI into its components:
Investment Fund → CDE → QALICB (A Loan + B Loan).


In [ ]:
tx_result = transaction.structure(deal)
tx_df = tx_result.summary()


## 3. 7-Year Tax Credit Schedule

NMTCs are earned at 5% of QEI in years 1-3 and 6% in years 4-7,
totaling 39% of the QEI over the compliance period.


In [ ]:
credit_result = credits.schedule(deal)
credit_df = credit_result.summary()

print(f"Total NMTCs:    \${credit_result.total_nmtcs/1e6:.2f}MM")
print(f"PV of Credits:  \${credit_result.pv_credits/1e6:.2f}MM @ {deal.discount_rate*100:.0f}% discount rate")


## 4. Investor Economics

The investor puts in equity upfront and receives tax credits
over 7 years. We compute IRR and MOIC on the investment.


In [ ]:
inv_result = investor.analyze(deal)
inv_df = inv_result.summary()

print(f"Investor Equity In:  \${inv_result.investor_equity/1e6:.2f}MM")
print(f"Total NMTCs Out:     \${inv_result.total_nmtcs/1e6:.2f}MM")
print(f"Net Benefit:         \${inv_result.net_benefit/1e6:.2f}MM")
print(f"MOIC:                {inv_result.moic:.2f}x")
print(f"IRR:                 {inv_result.irr*100:.1f}%")


## 5. Net Subsidy to the Project

The B Loan is typically forgiven via put/call at the end of the
7-year compliance period — this is the real economic subsidy
to the QALICB (the project borrower).


In [ ]:
sub_result = subsidy.analyze(deal)
sub_df = sub_result.summary()

print(f"Net Subsidy:               \${sub_result.net_subsidy/1e6:.2f}MM")
print(f"Net Subsidy as % of Cost:  {sub_result.net_subsidy_pct*100:.1f}%")
print(f"Effective Cost of Capital: {sub_result.effective_cost_of_capital*100:.2f}%")
print(f"Interest Savings (7yr):    \${sub_result.interest_savings_7yr/1e6:.2f}MM")


## Summary

| Item | Amount |
|------|--------|
| Total Project Cost | $12.0MM |
| NMTC Allocation (QEI) | $10.0MM |
| Total NMTCs (39%) | $3.9MM |
| Investor Equity | ~$3.2MM |
| Leverage Loan | ~$6.8MM |
| Net Subsidy to Project | ~$3.0MM (~25% of cost) |
| Effective Cost of Capital | <2% |

The NMTC structure delivers approximately 20-25% of project cost
as a permanent subsidy — enabling projects that would otherwise
be financially infeasible in low-income communities.

**GitHub:** https://github.com/Jaypatel1511/nmtc-calc  
**PyPI:** https://pypi.org/project/nmtc-calc
